# TUSZ (Google Drive -> EvoBrain 19-Ch 200Hz -> HuggingFace Hub)

This notebook reads the **Temple University Hospital EEG Seizure Corpus (TUSZ)** from your **Google Drive** (`edf/train`, `edf/dev`, `edf/eval`), extracts the exact **19 Unipolar Channels** at **200 Hz** required by **EvoBrain**,
attaches both **binary (`.csv_bi` / `.tse_bi`)** and **multi-class per-channel (`.csv`)** annotations, and packs the **full unsliced recordings** into **ZSTD-compressed Parquet shards** uploaded directly to HuggingFace Hub.

### Key Features for EvoBrain
- **Zero Disk Bloat (No Precomputing)**: Keeps recordings continuous `(19, T)` at 200 Hz instead of generating 250,000+ tiny `.h5` files.
- **EvoBrain 19-Channel 10-20 Referential Montage**: Matches `INCLUDED_CHANNELS` in EvoBrain (`FP1, FP2, F3, F4, C3, C4, P3, P4, O1, O2, F7, F8, T3, T4, T5, T6, FZ, CZ, PZ`).
- **Dynamic On-The-Fly FFT Slicing**: Slices 12s clips with 1s steps `(12, 19, 100)` directly in memory during training.
- **HuggingFace Shards OR Direct .h5**: Can stream from HuggingFace OR export standard EvoBrain `.h5` continuous files.

In [ ]:
# @title 1. Install Dependencies & Mount Google Drive
!pip install -q pyedflib mne scipy pyarrow datasets huggingface_hub tqdm matplotlib gdown google-api-python-client

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 1.5 (Optional) Load from a Google Drive Shared Link (Folder OR .zip/.tar.gz)
# If you have a Google Drive shared link (e.g. https://drive.google.com/drive/folders/... or .../file/d/...),
# paste it below!
# - If it's a SHARED FOLDER link: `gdown` fails after 50 files. Instead, this cell uses the Google Drive API
#   to automatically mount a 0-byte Shortcut into your `/content/drive/MyDrive/_TUSZ_Shared_Shortcut` so Colab
#   can read all 50,000+ files directly without downloading or using your Drive storage!
# - If it's a SHARED ARCHIVE (.zip / .tar.gz) link: it downloads & extracts it to `/content/TUSZ_local`.

import re
import os
import subprocess

GDRIVE_SHARED_LINK = ""  # @param {type:"string"}

if GDRIVE_SHARED_LINK.strip():
    link = GDRIVE_SHARED_LINK.strip()
    if "/folders/" in link or "?id=" in link:
        # Extract Folder ID from URL
        m = re.search(r"/folders/([a-zA-Z0-9_-]+)", link) or re.search(r"id=([a-zA-Z0-9_-]+)", link)
        folder_id = m.group(1) if m else link
        print(f"Detected Shared Google Drive FOLDER ID: {folder_id}")
        print("Creating automatic shortcut in your MyDrive via Google Drive API (uses 0 bytes of storage)...")

        from google.colab import auth
        from googleapiclient.discovery import build
        auth.authenticate_user()
        drive_service = build("drive", "v3")

        shortcut_name = "_TUSZ_Shared_Shortcut"
        # Check if shortcut already exists
        q = f"name = '{shortcut_name}' and mimeType = 'application/vnd.google-apps.shortcut' and trashed = false"
        res = drive_service.files().list(q=q, fields="files(id, name)").execute().get("files", [])
        if not res:
            shortcut_meta = {
                "name": shortcut_name,
                "mimeType": "application/vnd.google-apps.shortcut",
                "shortcutDetails": {"targetId": folder_id},
            }
            drive_service.files().create(body=shortcut_meta, fields="id").execute()
            # Flush Drive FUSE mount so the new shortcut appears immediately
            drive.flush_and_unmount()
            drive.mount('/content/drive')

        AUTO_TUSZ_ROOT = f"/content/drive/MyDrive/{shortcut_name}"
        print(f"Success! Shared folder is now mounted at: {AUTO_TUSZ_ROOT}")
    else:
        # Single archive file (.zip, .tar.gz, .tgz)
        os.makedirs("/content/TUSZ_local", exist_ok=True)
        print("Detected Shared File link. Downloading via gdown to /content/TUSZ_local ...")
        subprocess.run(["gdown", "--fuzzy", link, "-O", "/content/TUSZ_local/"], check=True)
        AUTO_TUSZ_ROOT = "/content/TUSZ_local"
        print("Downloaded to /content/TUSZ_local! (Extract if .zip/.tar.gz)")
else:
    AUTO_TUSZ_ROOT = None
    print("No shared link provided; will use GDRIVE_TUSZ_ROOT in Cell 2.")

In [ ]:
# @title 2. Configuration & HuggingFace Login
import os
from huggingface_hub import login, HfApi

# ======================== USER SETTINGS ========================
# Path to your TUSZ folder in Google Drive (automatically uses Shared Link shortcut if set in Cell 1.5!)
GDRIVE_TUSZ_ROOT = "/content/drive/MyDrive/TUSZ"  # @param {type:"string"}
if globals().get("AUTO_TUSZ_ROOT"):
    GDRIVE_TUSZ_ROOT = AUTO_TUSZ_ROOT
    print(f"Using Shared Link path: {GDRIVE_TUSZ_ROOT}")

# Your target HuggingFace Dataset Repo ID (e.g., "username/tusz-evobrain-19ch-200hz")
HF_REPO_ID = "your-username/tusz-evobrain-19ch-200hz"  # @param {type:"string"}
HF_TOKEN = ""                                          # @param {type:"string"}

    # Signal Processing Parameters (EvoBrain standard: 200 Hz, 19 channels)
    TARGET_SFREQ = 200          # Required by EvoBrain (200 Hz) {type:"raw"}
    APPLY_BANDPASS = True       # @param {type:"boolean"}
LOWCUT_HZ = 0.5             # @param {type:"number"}
HIGHCUT_HZ = 45.0           # @param {type:"number"}
CLIP_UV = 1000.0            # Clip extreme artifact spikes (+/- 1000 uV) so float16 stays high-precision

# Sharding & Subsampling Parameters
RECORDS_PER_SHARD = 250     # @param {type:"integer"}
NUM_WORKERS = 2             # Parallel file readers (keep 2-4 on Google Drive FUSE)
# Set to 1.0 to keep 100% of pure-background (no-seizure) files, or e.g. 0.25 to keep all seizure files + 25% of empty files
KEEP_EMPTY_FILE_RATIO = 1.0 # @param {type:"number"}
# ===============================================================

if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    login()  # Interactive prompt if token left blank

api = HfApi()
# IMPORTANT: Keep private=True to comply with Temple University Hospital (NEDC) data license!
api.create_repo(repo_id=HF_REPO_ID, repo_type="dataset", private=True, exist_ok=True)
print(f"Ready! Target HuggingFace Dataset Repo (Private): https://huggingface.co/datasets/{HF_REPO_ID}")

In [ ]:
# @title 3. Scan & Cache TUSZ File Index from Google Drive
import json
import random
from pathlib import Path
from tqdm.auto import tqdm

# Save index cache in your own MyDrive root so it works even if GDRIVE_TUSZ_ROOT is a read-only Shared Folder!
INDEX_CACHE_PATH = "/content/drive/MyDrive/_tusz_file_index_cache.json"

def find_split_dirs(root_dir: str):
    """Automatically locates train, dev, and eval directories inside GDRIVE_TUSZ_ROOT."""
    root = Path(root_dir)
    candidates = [root, root / "edf", root / "v2.0.0" / "edf", root / "v2.0.1" / "edf", root / "v1.5.2" / "edf"]
    for cand in candidates:
        if (cand / "train").is_dir():
            return {s: cand / s for s in ["train", "dev", "eval"] if (cand / s).is_dir()}
    # Fallback search
    split_dirs = {}
    for dirpath, dirnames, _ in os.walk(root):
        for s in ["train", "dev", "eval"]:
            if s in dirnames and s not in split_dirs:
                split_dirs[s] = Path(dirpath) / s
        if len(split_dirs) == 3:
            break
    return split_dirs

if os.path.exists(INDEX_CACHE_PATH):
    print(f"Loading cached file index from {INDEX_CACHE_PATH} ...")
    with open(INDEX_CACHE_PATH, "r") as f:
        dataset_index = json.load(f)
else:
    split_dirs = find_split_dirs(GDRIVE_TUSZ_ROOT)
    assert len(split_dirs) > 0, f"Could not find train/dev/eval inside {GDRIVE_TUSZ_ROOT}! Check your path."
    print("Found split directories:", {k: str(v) for k, v in split_dirs.items()})

    dataset_index = {}
    for split, sdir in split_dirs.items():
        records = []
        print(f"Scanning '{split}' split in {sdir} (this takes 1-3 mins the first time on Google Drive)...")
        for root, _, files in tqdm(os.walk(sdir), desc=f"Walking {split}"):
            edf_files = [f for f in files if f.lower().endswith(".edf")]
            for edf_name in edf_files:
                base = edf_name[:-4]
                edf_path = os.path.join(root, edf_name)
                csv_path = os.path.join(root, base + ".csv")
                csv_bi_path = os.path.join(root, base + ".csv_bi")
                tse_path = os.path.join(root, base + ".tse")
                tse_bi_path = os.path.join(root, base + ".tse_bi")

                records.append({
                    "record_id": base,
                    "split": split,
                    "edf_path": edf_path,
                    "csv_path": csv_path if os.path.exists(csv_path) else (tse_path if os.path.exists(tse_path) else ""),
                    "csv_bi_path": csv_bi_path if os.path.exists(csv_bi_path) else (tse_bi_path if os.path.exists(tse_bi_path) else ""),
                })
        records.sort(key=lambda x: x["record_id"])
        dataset_index[split] = records

    with open(INDEX_CACHE_PATH, "w") as f:
        json.dump(dataset_index, f)
    print(f"Saved index cache to {INDEX_CACHE_PATH}")

for split, recs in dataset_index.items():
    print(f"  Split '{split}': {len(recs):,} EDF recordings")

In [ ]:
# @title 4. Define 19-Channel Referential Montage (EvoBrain Standard) & Annotation Parser
import re
import numpy as np
import pyedflib
from fractions import Fraction
from scipy.signal import butter, filtfilt, resample_poly

# Standard 19 Referential Channels expected by EvoBrain (matching constants.py INCLUDED_CHANNELS)
EVOBRAIN_19_CHANNELS = [
    "FP1", "FP2", "F3", "F4", "C3", "C4",
    "P3", "P4", "O1", "O2", "F7", "F8",
    "T3", "T4", "T5", "T6", "FZ", "CZ", "PZ"
]

# Modern 10-10 fallback aliases (e.g., T7/T8/P7/P8 or Fp1 lowercase)
ELECTRODE_ALIASES = {"T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8", "FP1": "FP1", "FP2": "FP2"}

def normalize_edf_label(label: str) -> str:
    """Converts 'EEG FP1-REF', 'EEG FP1-LE', or 'EEG FP1' -> 'FP1'."""
    s = label.upper().strip()
    s = re.sub(r"^EEG\s+", "", s)
    s = re.sub(r"-(REF|LE)$", "", s)
    return s.strip()

def parse_tusz_annotations(ann_path: str):
    """Parses TUSZ v2.0+ (.csv / .csv_bi) or legacy v1.5.2 (.tse / .tse_bi) files."""
    events = []
    duration = None
    if not ann_path or not os.path.exists(ann_path):
        return events, duration

    with open(ann_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("#"):
                if "duration" in line.lower():
                    m = re.search(r"([0-9]+\.?[0-9]*)", line)
                    if m:
                        duration = float(m.group(1))
                continue
            if line.lower().startswith("channel,") or line.lower().startswith("version"):
                continue

            # CSV format: channel,start_time,stop_time,label,confidence
            if "," in line:
                parts = [p.strip() for p in line.split(",")]
                if len(parts) >= 4:
                    if len(parts) >= 6 and not parts[1].replace(".", "", 1).isdigit():
                        ch, start, stop, lbl = parts[1], float(parts[2]), float(parts[3]), parts[4].lower()
                    else:
                        ch, start, stop, lbl = parts[0], float(parts[1]), float(parts[2]), parts[3].lower()
                    events.append({"channel": ch.upper(), "start": start, "stop": stop, "label": lbl})
            else:
                # Legacy .tse format: start_time stop_time label probability
                parts = line.split()
                if len(parts) >= 3 and parts[0].replace(".", "", 1).isdigit():
                    events.append({
                        "channel": "TERM",
                        "start": float(parts[0]),
                        "stop": float(parts[1]),
                        "label": parts[2].lower()
                    })
    return events, duration

def process_single_recording(rec_meta: dict, target_sfreq: int = TARGET_SFREQ):
    """
    Reads an EDF file, extracts the 19 unipolar 10-20 channels required by EvoBrain,
    filters & resamples to target_sfreq (200 Hz), casts to float16, and packages metadata + annotations.
    """
    edf_path = rec_meta["edf_path"]
    f = pyedflib.EdfReader(edf_path)
    try:
        raw_labels = f.getSignalLabels()
        ch_map = {}
        for idx, lbl in enumerate(raw_labels):
            norm = normalize_edf_label(lbl)
            if norm not in ch_map:
                ch_map[norm] = idx

        unipolar_signals = {}
        unipolar_sfreqs = {}
        for elec in EVOBRAIN_19_CHANNELS:
            alias = ELECTRODE_ALIASES.get(elec, "")
            idx = ch_map.get(elec, ch_map.get(alias))
            if idx is not None:
                sig = f.readSignal(idx).astype(np.float32)
                dim = f.getPhysicalDimension(idx).strip().lower()
                if dim == "mv":
                    sig *= 1000.0
                elif dim == "v":
                    sig *= 1e6
                unipolar_signals[elec] = sig
                unipolar_sfreqs[elec] = float(f.getSampleFrequency(idx))
    finally:
        f._close()

    if not unipolar_signals:
        raise ValueError(f"No valid 10-20 channels found in {edf_path}")

    orig_sfreq = sorted(unipolar_sfreqs.values())[len(unipolar_sfreqs) // 2]
    min_len = min(len(s) for s in unipolar_signals.values() if len(s) > 0)

    # Extract the 19 unipolar channels in exact EvoBrain order
    channel_list = []
    channel_mask = []
    for elec in EVOBRAIN_19_CHANNELS:
        if elec in unipolar_signals:
            channel_list.append(unipolar_signals[elec][:min_len])
            channel_mask.append(True)
        else:
            # Fallback zero-pad if one channel was disconnected
            channel_list.append(np.zeros(min_len, dtype=np.float32))
            channel_mask.append(False)

    eeg_data = np.stack(channel_list, axis=0)  # Shape: (19, min_len)

    # Optional Bandpass Filter (0.5 - 45 Hz)
    if APPLY_BANDPASS and min_len > int(orig_sfreq * 2):
        nyq = 0.5 * orig_sfreq
        low = max(LOWCUT_HZ / nyq, 1e-4)
        high = min(HIGHCUT_HZ / nyq, 0.99)
        if low < high:
            b_filt, a_filt = butter(4, [low, high], btype="band")
            eeg_data = filtfilt(b_filt, a_filt, eeg_data, axis=-1).astype(np.float32)

    # Resample to TARGET_SFREQ (200 Hz for EvoBrain)
    if abs(orig_sfreq - target_sfreq) > 1e-3:
        frac = Fraction(int(round(target_sfreq)), int(round(orig_sfreq))).limit_denominator(1000)
        eeg_data = resample_poly(eeg_data, frac.numerator, frac.denominator, axis=-1).astype(np.float32)

    # Clip extreme spikes & cast to float16 for storage efficiency
    np.clip(eeg_data, -CLIP_UV, CLIP_UV, out=eeg_data)
    eeg_fp16 = eeg_data.astype(np.float16)

    events_bi, dur_bi = parse_tusz_annotations(rec_meta["csv_bi_path"])
    events_multi, dur_multi = parse_tusz_annotations(rec_meta["csv_path"])
    duration_sec = float(eeg_fp16.shape[1]) / float(target_sfreq)

    seiz_intervals_bi = [
        {"start": e["start"], "stop": e["stop"], "label": e["label"]}
        for e in events_bi if e["label"] != "bckg"
    ]
    seiz_intervals_multi = [
        {"channel": e["channel"], "start": e["start"], "stop": e["stop"], "label": e["label"]}
        for e in events_multi if e["label"] != "bckg"
    ]
    seizure_types = sorted(list({e["label"] for e in seiz_intervals_multi} | {e["label"] for e in seiz_intervals_bi if e["label"] != "seiz"}))
    has_seizure = len(seiz_intervals_bi) > 0 or len(seiz_intervals_multi) > 0

    parts = rec_meta["record_id"].split("_")
    patient_id = parts[0] if len(parts) > 0 else "unknown"
    session_id = parts[1] if len(parts) > 1 else "s001"
    token_id = parts[2] if len(parts) > 2 else "t000"
    montage = Path(edf_path).parent.name

    return {
        "record_id": rec_meta["record_id"],
        "split": rec_meta["split"],
        "patient_id": patient_id,
        "session_id": session_id,
        "token_id": token_id,
        "montage": montage,
        "sfreq": int(target_sfreq),
        "orig_sfreq": float(orig_sfreq),
        "n_channels": 19,
        "n_samples": int(eeg_fp16.shape[1]),
        "duration_sec": float(duration_sec),
        "channel_mask": channel_mask,
        "has_seizure": bool(has_seizure),
        "seizure_types": seizure_types,
        "events_bi_json": json.dumps(seiz_intervals_bi),
        "events_multi_json": json.dumps(seiz_intervals_multi),
        "eeg_fp16_bytes": eeg_fp16.tobytes(),
    }

print("EvoBrain 19-Channel 200Hz processor ready!")

In [ ]:
# @title 5. Sanity Check & Visualize 1 Seizure Recording from Your Google Drive
import matplotlib.pyplot as plt

# Find the first recording in 'train' or 'dev' that has a seizure in .csv_bi
sample_rec = None
for split_name in ["train", "dev", "eval"]:
    for r in dataset_index.get(split_name, []):
        ev_bi, _ = parse_tusz_annotations(r["csv_bi_path"])
        seiz_ev = [e for e in ev_bi if e["label"] != "bckg" and e["start"] > 15.0]
        if seiz_ev:
            sample_rec = r
            break
    if sample_rec:
        break

if sample_rec is None:
    sample_rec = dataset_index[list(dataset_index.keys())[0]][0]

processed = process_single_recording(sample_rec)
raw_size_kb = os.path.getsize(sample_rec["edf_path"]) / 1024.0
fp16_size_kb = len(processed["eeg_fp16_bytes"]) / 1024.0

print(f"Sample Record   : {processed['record_id']} (Patient: {processed['patient_id']}, Montage: {processed['montage']})")
print(f"Duration        : {processed['duration_sec']:.1f} sec | Orig Hz: {processed['orig_sfreq']} -> Resampled Hz: {processed['sfreq']}")
print(f"Has Seizure     : {processed['has_seizure']} | Subtypes: {processed['seizure_types']}")
print(f"Binary Events   : {processed['events_bi_json']}")
print(f"Size Reduction  : {raw_size_kb:,.1f} KB (.edf) -> {fp16_size_kb:,.1f} KB (raw float16 before ZSTD compression)")

# Reconstruct (19, T) numpy array from bytes in 0ms
eeg = np.frombuffer(processed["eeg_fp16_bytes"], dtype=np.float16).reshape(19, processed["n_samples"]).astype(np.float32)
events_bi = json.loads(processed["events_bi_json"])

# Plot a 30-second window centered on the first seizure onset
onset_sec = events_bi[0]["start"] if events_bi else min(15.0, processed["duration_sec"] / 2)
win_start = max(0.0, onset_sec - 15.0)
win_end = min(processed["duration_sec"], onset_sec + 15.0)
s_idx = int(win_start * processed["sfreq"])
e_idx = int(win_end * processed["sfreq"])
t_axis = np.linspace(win_start, win_end, e_idx - s_idx)

fig, ax = plt.subplots(figsize=(15, 9))
offset = 150.0  # uV spacing between channels
for ch_i in range(19):
    ax.plot(t_axis, eeg[ch_i, s_idx:e_idx] + (18 - ch_i) * offset, lw=0.7, color="#1f2937")

if events_bi:
    ax.axvline(onset_sec, color="#dc2626", linestyle="--", lw=2, label=f"Seizure Onset ({onset_sec:.1f}s)")
    ax.axvspan(win_start, onset_sec, color="#fef08a", alpha=0.25, label="Near-Future Pre-Ictal Window")
    ax.axvspan(onset_sec, min(win_end, events_bi[0]["stop"]), color="#fecaca", alpha=0.35, label=f"Ictal / Seizure ({processed['seizure_types']})")

ax.set_yticks([(18 - i) * offset for i in range(19)])
ax.set_yticklabels(EVOBRAIN_19_CHANNELS, fontsize=8)
ax.set_xlabel("Time (seconds)", fontsize=11)
ax.set_title(f"TUSZ 19-Channel Referential Montage (EvoBrain): {processed['record_id']} ({processed['sfreq']} Hz, float16)", fontsize=13)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# @title 6. Build Compressed Parquet Shards & Upload to HuggingFace (Resume-Safe)
import gc
import pyarrow as pa
import pyarrow.parquet as pq
from concurrent.futures import ThreadPoolExecutor

LOCAL_SHARD_DIR = "/content/tmp_tusz_shards"
os.makedirs(LOCAL_SHARD_DIR, exist_ok=True)

# Explicit PyArrow Schema for fast ZSTD Parquet serialization
PARQUET_SCHEMA = pa.schema([
    ("record_id", pa.string()),
    ("split", pa.string()),
    ("patient_id", pa.string()),
    ("session_id", pa.string()),
    ("token_id", pa.string()),
    ("montage", pa.string()),
    ("sfreq", pa.int32()),
    ("orig_sfreq", pa.float32()),
    ("n_channels", pa.int32()),
    ("n_samples", pa.int32()),
    ("duration_sec", pa.float32()),
    ("channel_mask", pa.list_(pa.bool_())),
    ("has_seizure", pa.bool_()),
    ("seizure_types", pa.list_(pa.string())),
    ("events_bi_json", pa.string()),
    ("events_multi_json", pa.string()),
    ("eeg_fp16_bytes", pa.binary()),
])

# Check which shards already exist on HuggingFace so we can resume seamlessly
existing_hf_files = set(api.list_repo_files(repo_id=HF_REPO_ID, repo_type="dataset"))
print(f"Found {len(existing_hf_files)} existing files in {HF_REPO_ID}.")

def safe_process(rec_meta):
    try:
        # Optional fast filter before reading EDF if subsampling empty background files
        if KEEP_EMPTY_FILE_RATIO < 1.0:
            ev_bi, _ = parse_tusz_annotations(rec_meta["csv_bi_path"])
            has_seiz = any(e["label"] != "bckg" for e in ev_bi)
            if not has_seiz:
                # Deterministic hash so resume produces the exact same subset
                if (hash(rec_meta["record_id"]) % 10000) / 10000.0 > KEEP_EMPTY_FILE_RATIO:
                    return None
        return process_single_recording(rec_meta, target_sfreq=TARGET_SFREQ)
    except Exception as e:
        print(f"[WARN] Skipping corrupted file {rec_meta['edf_path']}: {e}")
        return None

for split, records in dataset_index.items():
    num_shards = (len(records) + RECORDS_PER_SHARD - 1) // RECORDS_PER_SHARD
    print(f"\n================ Processing Split: {split} ({len(records)} files -> {num_shards} shards) ================")

    for shard_idx in range(num_shards):
        hf_path_in_repo = f"data/{split}/{split}-shard-{shard_idx:04d}-of-{num_shards:04d}.parquet"
        if hf_path_in_repo in existing_hf_files:
            print(f"  [SKIP] {hf_path_in_repo} already uploaded to HuggingFace.")
            continue

        batch_meta = records[shard_idx * RECORDS_PER_SHARD : (shard_idx + 1) * RECORDS_PER_SHARD]
        rows = []

        with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
            for res in tqdm(pool.map(safe_process, batch_meta), total=len(batch_meta), desc=f"{split} shard {shard_idx+1}/{num_shards}"):
                if res is not None:
                    rows.append(res)

        if not rows:
            continue

        # Write ZSTD-compressed Parquet shard locally
        local_shard_path = os.path.join(LOCAL_SHARD_DIR, f"{split}-shard-{shard_idx:04d}.parquet")
        table = pa.Table.from_pylist(rows, schema=PARQUET_SCHEMA)
        pq.write_table(table, local_shard_path, compression="zstd", compression_level=5)

        shard_mb = os.path.getsize(local_shard_path) / (1024 * 1024)
        print(f"  Uploading {hf_path_in_repo} ({len(rows)} recordings, {shard_mb:.1f} MB)...")

        api.upload_file(
            path_or_fileobj=local_shard_path,
            path_in_repo=hf_path_in_repo,
            repo_id=HF_REPO_ID,
            repo_type="dataset",
            commit_message=f"Upload {hf_path_in_repo} ({len(rows)} records)"
        )

        # Delete local shard immediately to keep Colab disk free
        os.remove(local_shard_path)
        del rows, table
        gc.collect()

print(f"\nAll shards uploaded to https://huggingface.co/datasets/{HF_REPO_ID} !")

In [ ]:
# @title 7. EvoBrain Dynamic Windowing & On-The-Fly FFT Slicing (Zero Disk Waste)
import torch
import numpy as np
import json
from scipy.fftpack import fft
from torch.utils.data import Dataset
from datasets import load_dataset

def compute_step_fft(signal_1s, n_fft=200):
    """
    Computes log-magnitude FFT on 1-second segment (200 samples) across 19 channels.
    Matches computeFFT in EvoBrain/data/data_utils.py.
    Returns: (19, 100) array of positive frequency log-amplitudes.
    """
    fourier_signal = fft(signal_1s, n=n_fft, axis=-1)
    idx_pos = int(np.floor(n_fft / 2))  # 100 positive frequency bins
    amp = np.abs(fourier_signal[:, :idx_pos])
    amp[amp == 0.0] = 1e-8
    return np.log(amp)

class EvoBrainDynamicDataset(Dataset):
    """
    Slices 12-second clips (or any clip length) on-the-fly and computes the
    100-bin positive frequency FFT per 1-second step directly in RAM.
    Produces tensor shape: (clip_len, 19, 100) exactly as expected by EvoBrain!
    Consumes ZERO extra disk space (no 250,000 precomputed .h5 files).
    """
    def __init__(self, hf_dataset, clip_len_sec: int = 12, stride_sec: int = 1, is_fft: bool = True):
        self.hf_ds = hf_dataset
        self.clip_len_sec = clip_len_sec
        self.stride_sec = stride_sec
        self.is_fft = is_fft
        self.windows = []

        ds_meta = self.hf_ds.remove_columns(["eeg_fp16_bytes"])
        for row_idx in range(len(ds_meta)):
            row = ds_meta[row_idx]
            sfreq = row["sfreq"]
            n_samples = row["n_samples"]
            win_samples = int(clip_len_sec * sfreq)
            stride_samples = int(stride_sec * sfreq)
            if n_samples < win_samples:
                continue

            events_bi = json.loads(row["events_bi_json"])
            for start_samp in range(0, n_samples - win_samples + 1, stride_samples):
                end_samp = start_samp + win_samples
                w_start = start_samp / sfreq
                w_end = end_samp / sfreq

                # Binary label: 1 if window overlaps an active seizure, else 0
                is_seizure = 0
                for ev in events_bi:
                    if max(w_start, ev["start"]) < min(w_end, ev["stop"]):
                        is_seizure = 1
                        break

                self.windows.append((row_idx, start_samp, end_samp, is_seizure))

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        row_idx, start_samp, end_samp, is_seizure = self.windows[idx]
        row = self.hf_ds[row_idx]
        sfreq = row["sfreq"]

        # Decode 19-channel unsliced continuous recording from buffer in 0ms
        eeg_full = np.frombuffer(row["eeg_fp16_bytes"], dtype=np.float16).reshape(19, row["n_samples"])
        clip_raw = eeg_full[:, start_samp:end_samp].astype(np.float32)  # (19, 2400)

        if self.is_fft:
            # Slices into 1-second steps and computes 100-bin positive FFT
            fft_steps = []
            step_samples = int(sfreq * 1)  # 200 samples per second
            for t in range(self.clip_len_sec):
                step_data = clip_raw[:, t * step_samples : (t + 1) * step_samples]
                step_fft = compute_step_fft(step_data, n_fft=step_samples)  # (19, 100)
                fft_steps.append(step_fft)
            x = np.stack(fft_steps, axis=0)  # Shape: (12, 19, 100)
        else:
            x = clip_raw

        return {
            "x": torch.from_numpy(x).float(),            # (12, 19, 100) -> EvoBrain input shape
            "y": torch.tensor([is_seizure]).float(),      # Seizure ground truth (0 or 1)
            "seq_len": torch.tensor([self.clip_len_sec]), # (12,)
            "record_id": row["record_id"]
        }

print("EvoBrainDynamicDataset defined! Zero precomputed files needed.")

# =====================================================================
# (Optional) Export Continuous .h5 Files Directly to Google Drive
# If you prefer running EvoBrain main.py locally, export unsliced .h5 files:
# =====================================================================
import h5py

def export_continuous_evobrain_h5(record_meta, eeg_fp16, out_dir="/content/drive/MyDrive/TUSZ_EvoBrain_resampled"):
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, record_meta["record_id"] + ".h5")
    eeg_fp32 = eeg_fp16.astype(np.float32)
    with h5py.File(out_path, "w") as hf:
        hf.create_dataset("resampled_signal", data=eeg_fp32)
        hf.create_dataset("resample_freq", data=200)
